# Parse price list PDF

Extract text from a PDF (e.g. price list or quote) using `pdfplumber`.

## Setup

Install `pdfplumber` in the kernel if needed: run `%pip install pdfplumber` once.

In [42]:
from pathlib import Path

# Price list paths (relative to notebook / validation_quotes)
PATH_2026 = Path("2026/Price list 2026 - 32 percent discount.pdf")
PATH_PRE2026 = Path("pre2026 Quotes/new 2024 VP Price list 23pg Sept 27-1.pdf")

# Toggle which price list to use
USE_2026 = True  # False = use pre2026 (2024) list

if USE_2026:
    PDF_PATH = PATH_2026
    MAX_PAGE = 15  # 2026 list: windows through page 15
    PRICE_LIST_SUFFIX = "2026"
else:
    PDF_PATH = PATH_PRE2026
    MAX_PAGE = 12  # 2024 list: windows through page 12
    PRICE_LIST_SUFFIX = "pre2026"

print(f"Using: {PDF_PATH} (pages 1–{MAX_PAGE} for extraction/prompt), suffix={PRICE_LIST_SUFFIX}")

Using: 2026/Price list 2026 - 32 percent discount.pdf (pages 1–15 for extraction/prompt), suffix=2026


In [43]:
import pdfplumber
with pdfplumber.open(PDF_PATH) as pdf:
    print(f"Pages: {len(pdf.pages)}")
    for i, page in enumerate(pdf.pages):
        print(f"  {i + 1}: {page.width} x {page.height}")

Pages: 27
  1: 612.0 x 792.0
  2: 612.0 x 792.0
  3: 612.0 x 792.0
  4: 612.0 x 792.0
  5: 612.0 x 792.0
  6: 612.0 x 792.0
  7: 612.0 x 792.0
  8: 612.0 x 792.0
  9: 612.0 x 792.0
  10: 612.0 x 792.0
  11: 612.0 x 792.0
  12: 612.0 x 792.0
  13: 612.0 x 792.0
  14: 612.0 x 792.0
  15: 612.0 x 792.0
  16: 612.0 x 792.0
  17: 612.0 x 792.0
  18: 612.0 x 792.0
  19: 612.0 x 792.0
  20: 612.0 x 792.0
  21: 612.0 x 792.0
  22: 612.0 x 792.0
  23: 612.0 x 792.0
  24: 612.0 x 792.0
  25: 612.0 x 792.0
  26: 612.0 x 792.0
  27: 612.0 x 792.0


## Extract text (up to MAX_PAGE)

In [44]:
parts = []
with pdfplumber.open(PDF_PATH) as pdf:
    for i, page in enumerate(pdf.pages):
        if i >= MAX_PAGE:
            break
        text = page.extract_text()
        parts.append(f"--- Page {i + 1} ---\n\n{text or '(no text)'}")
parts = parts[:MAX_PAGE]
full_text = "\n\n".join(parts)
print(f"Combined text: {len(full_text)} chars, {len(parts)} pages (max page {MAX_PAGE})")

Combined text: 6315 chars, 15 pages (max page 15)


In [45]:
with open(f"price_list_text_{PRICE_LIST_SUFFIX}.txt", "w", encoding="utf-8") as f:
    f.write(full_text)
print(f"Wrote price_list_text_{PRICE_LIST_SUFFIX}.txt")

Wrote price_list_text_2026.txt


## Send to model for pricing JSON
Use only **pages 1–MAX_PAGE** (windows only; no glass, shapes, extensions, brickmould, grills, labour). Model returns JSON with window-type keys that include the high-level measurement (e.g. `4_9_16_casement`).

In [48]:
import sys
from pathlib import Path
# Find project root (directory containing llm_io) by walking up from cwd
_cwd = Path().resolve()
_project_root = None
for p in [_cwd] + list(_cwd.parents):
    if (p / "llm_io").is_dir():
        _project_root = p
        break
if _project_root is None:
    raise RuntimeError("Project root (folder containing llm_io) not found. Run the notebook from ai-estimator or a subfolder.")
if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

from llm_io.model_io import ModelIO

if PRICE_LIST_SUFFIX == "2026":
    PRICING_JSON_PROMPT = """You are given the text of the 2026 window dealer price list PDF (pages 1-15 only, with page dividers '--- Page N ---').
Extract window pricing only into a single YAML document. Do NOT include glass, labour, shapes, extensions, brickmould, grills, or anything after the window pages. You must include every window product that appears in these pages.

Use this YAML structure for each window type (keys and nesting only; values come from the PDF):

casement:
  white:
    - max_sf: <number>
      price: <number>
      per_sf_rate: <number or 0 if not shown>
    - max_sf: <number>
      price: <number>
      per_sf_rate: <number or 0 if not shown>
  colour:
    - max_sf: <number>
      price: <number>
      per_sf_rate: <number or 0 if not shown>
  exterior:
    colour_base_perc: <decimal, e.g. 0.25>
    custom_colour_add_on: <number>
    stain_add_on: <number>
  interior:
    stain_add_on: <number>

- Top-level keys = window types. Use ONLY the measurement prefix 4_9_16_ when the section is explicitly 4-9/16 thickness (e.g. 4_9_16_casement, 4_9_16_awning, 4_9_16_fixed_window, 4_9_16_picture_window). For 3-1/4 or any other thickness, do NOT include a measurement prefix: use just the product type in snake_case (casement, awning, single_slider_tilt_out, double_hung_tilt, etc.). Never use 3_1_4_ in the key.
- For each window type, under white and colour:
  - Create a list of tiers. Each tier must have:
    - max_sf: the maximum square feet for that tier.
    - price: the price shown for that tier.
    - per_sf_rate: the per-square-foot rate for the "Over X Sq.Ft" row; use 0 on tiers where only a fixed price is shown.
- VERY IMPORTANT: When stain pricing is shown, ALWAYS populate BOTH:
  - exterior.stain_add_on with the EXTERIOR stain amount (or the relevant under/over amount if the table shows different stain prices by size),
  - interior.stain_add_on with the INTERIOR stain amount.
  Never omit exterior or interior stain_add_on when they appear in the PDF.
- Also always include:
  - exterior.colour_base_perc as a decimal (e.g. 0.25 for 25%),
  - exterior.custom_colour_add_on with the custom colour add-on amount.

Return ONLY valid YAML (no JSON, no markdown, no explanation)."""
else:
    PRICING_JSON_PROMPT = """You are given the text of the 2024 window dealer price list PDF (pages 1-12 only, with page dividers '--- Page N ---').
Extract window pricing only into a single JSON object. Do NOT include glass, labour, shapes, extensions, brickmould, grills, or anything after the window pages. You must include every window product that appears in these pages.

- Top-level keys = window types. Include the high-level 4-9/16 measurement as 4_9_16_ when the section is 4-9/16 (e.g. 4_9_16_casement, 4_9_16_awning, 4_9_16_fixed_casement, 4_9_16_picture_window). For 3-1/4 sections, use the product type alone in snake_case (casement, awning, single_slider_tilt_out, double_hung_tilt, etc.). Exception: on page 8, '4-9/16 casement' is a typo — use the key 'casement' only (no measurement prefix).
- For each window type: include 'white' and 'colour' (or 'interior' where the PDF shows only one finish) as lists of tiers: {"max_sf": number, "price": number} and an optional {"per_sf_rate": number}. "Over X Sq.Ft" rows -> per_sf_rate only; base tiers have price.
- When stain pricing is shown, always extract BOTH:
  - exterior.stain_add_on (the exterior stain price, or exterior.stain_add_on_under_14_sf / exterior.stain_add_on_over_14_sf where under/over 14 sq.ft is shown),
  - interior.stain_add_on (the interior stain price, or interior.stain_add_on_under_14_sf / interior.stain_add_on_over_14_sf where under/over 14 sq.ft is shown).
- Also include exterior.colour_base_perc (as decimal, e.g. 0.25) and exterior.custom_colour_add_on where present.
Return only valid JSON, no markdown or explanation."""

# Only send pages 1-MAX_PAGE to the model (windows only; no extensions, glass, etc.)
text_for_model = "\n\n".join(parts[:MAX_PAGE])
model_io = ModelIO("openai", "gpt-4o", PRICING_JSON_PROMPT)
response_text = model_io.get_response(message=text_for_model) or ""
print("Model response (first 2000 chars):")
print(response_text[:2000])

Model response (first 2000 chars):
```yaml
4_9_16_casement:
  white:
    - max_sf: 6
      price: 231.66
      per_sf_rate: 0
    - max_sf: 9
      price: 261.74
      per_sf_rate: 0
    - max_sf: 12
      price: 291.99
      per_sf_rate: 0
    - max_sf: 1000
      price: 0
      per_sf_rate: 24.38
  colour:
    - max_sf: 6
      price: 314.16
      per_sf_rate: 0
    - max_sf: 9
      price: 344.24
      per_sf_rate: 0
    - max_sf: 12
      price: 374.49
      per_sf_rate: 0
    - max_sf: 1000
      price: 0
      per_sf_rate: 32.91
  exterior:
    colour_base_perc: 0.25
    custom_colour_add_on: 350
    stain_add_on: 126
  interior:
    stain_add_on: 170

4_9_16_awning:
  white:
    - max_sf: 6
      price: 249.29
      per_sf_rate: 0
    - max_sf: 9
      price: 279.54
      per_sf_rate: 0
    - max_sf: 12
      price: 309.80
      per_sf_rate: 0
    - max_sf: 1000
      price: 0
      per_sf_rate: 26.55
  colour:
    - max_sf: 6
      price: 331.88
      per_sf_rate: 0
    - max_s

## Price list as YAML
Write the extracted pricing to `price_list_extracted_<suffix>.yaml` (suffix = 2026 or pre2026) for inspection or use as a pricing config.

In [47]:
import json
import re
import yaml

# Parse model response (strip markdown fence, then YAML)
raw = response_text.strip()
if raw.startswith("```"):
    raw = re.sub(r"^```\w*\n?", "", raw)
    raw = re.sub(r"\n?```\s*$", "", raw)
pricing_json = yaml.safe_load(raw)

def _rename_interior_paint(obj):
    """Recursively rename key 'interior_paint' to 'interior_color'."""
    if isinstance(obj, dict):
        return {(k if k != "interior_paint" else "interior_color"): _rename_interior_paint(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_rename_interior_paint(x) for x in obj]
    return obj

pricing_for_yaml = _rename_interior_paint(pricing_json)
yaml_str = yaml.dump(pricing_for_yaml, default_flow_style=False, sort_keys=False, allow_unicode=True)
with open(f"price_list_extracted_{PRICE_LIST_SUFFIX}.yaml", "w", encoding="utf-8") as f:
    f.write(yaml_str)
print(f"Wrote price_list_extracted_{PRICE_LIST_SUFFIX}.yaml ({len(pricing_json)} window types)")
print("\n--- Preview ---")
print(yaml_str[:3000])

Wrote price_list_extracted_2026.yaml (16 window types)

--- Preview ---
4_9_16_casement:
  white:
  - max_sf: 6
    price: 231.66
  - max_sf: 9
    price: 261.74
  - max_sf: 12
    price: 291.99
  - per_sf_rate: 24.38
  colour:
  - max_sf: 6
    price: 314.16
  - max_sf: 9
    price: 344.24
  - max_sf: 12
    price: 374.49
  - per_sf_rate: 32.91
4_9_16_awning:
  white:
  - max_sf: 6
    price: 249.29
  - max_sf: 9
    price: 279.54
  - max_sf: 12
    price: 309.8
  - per_sf_rate: 26.55
  colour:
  - max_sf: 6
    price: 331.88
  - max_sf: 9
    price: 369.75
  - max_sf: 12
    price: 410.42
  - per_sf_rate: 34.73
4_9_16_fixed_window:
  white:
  - max_sf: 7
    price: 146.0
  - per_sf_rate: 20.51
  colour:
  - max_sf: 7
    price: 206.0
  - per_sf_rate: 27.68
4_9_16_picture_window:
  white:
  - max_sf: 7
    price: 131.16
  - per_sf_rate: 18.87
  colour:
  - max_sf: 7
    price: 191.16
  - per_sf_rate: 25.47
4_9_16_single_slider_tilt_out:
  white:
  - max_sf: 6
    price: 190.5
  - max_